In [ ]:
%pip install dbx_test

In [ ]:
dbutils.library.restartPython()

In [ ]:
from dbx_test import NotebookTestFixture, NotebookRunner
from dbx_test.pytest_compat import parametrize, skipif, xfail
from dbx_test.fixtures import notebook_fixture
from pyspark.sql.functions import col, count, lit
from pyspark.sql.types import LongType, StringType, TimestampType

In [ ]:
# ============================================================================
# Configuration
# Catalog and schema are set via bundle spark conf; fall back to defaults.
# ============================================================================

try:
    CATALOG = spark.conf.get("catalog")
    SCHEMA = spark.conf.get("schema")
except:
    CATALOG = "serverless_stable_az5k2q_catalog"
    SCHEMA = "staging_kevin_potter"

VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/raw_data"

print(f"Testing pipeline output in {CATALOG}.{SCHEMA}")
print(f"Source volume: {VOLUME_PATH}")

In [ ]:
# ============================================================================
# Fixtures
# ============================================================================

@notebook_fixture
def catalog():
    return CATALOG

@notebook_fixture
def schema():
    return SCHEMA

@notebook_fixture
def fqn():
    """Returns a function to build fully qualified table names."""
    def _fqn(table):
        return f"{CATALOG}.{SCHEMA}.{table}"
    return _fqn

In [ ]:
# ============================================================================
# Test: Source Data Exists in Volume
# ============================================================================

class TestSourceData(NotebookTestFixture):
    """Validate that raw CDC data was seeded into the volume."""

    def run_setup(self):
        self.customers_path = f"{VOLUME_PATH}/customers"
        self.transactions_path = f"{VOLUME_PATH}/transactions"

    def test_customers_source_not_empty(self):
        df = spark.read.json(self.customers_path)
        assert df.count() > 0, "No customer source data in volume"

    def test_transactions_source_not_empty(self):
        df = spark.read.json(self.transactions_path)
        assert df.count() > 0, "No transaction source data in volume"

    def test_customers_source_has_expected_columns(self):
        df = spark.read.json(self.customers_path)
        expected = {"id", "firstname", "lastname", "email", "address", "operation", "operation_date"}
        actual = set(df.columns)
        missing = expected - actual
        assert not missing, f"Missing source columns: {missing}"

    def test_transactions_source_has_expected_columns(self):
        df = spark.read.json(self.transactions_path)
        expected = {"id", "transaction_date", "amount", "item_count", "operation", "operation_date", "customer_id"}
        actual = set(df.columns)
        missing = expected - actual
        assert not missing, f"Missing source columns: {missing}"

In [ ]:
# ============================================================================
# Test: Bronze Layer (Raw CDC Ingestion)
# ============================================================================

class TestBronzeLayer(NotebookTestFixture):
    """Validate that Auto Loader ingested raw CDC data into bronze tables."""

    def run_setup(self):
        self.customers_cdc = spark.table(f"{CATALOG}.{SCHEMA}.customers_cdc")
        self.transactions_cdc = spark.table(f"{CATALOG}.{SCHEMA}.transactions_cdc")

    @parametrize("table_attr", [("customers_cdc",), ("transactions_cdc",)])
    def test_bronze_table_not_empty(self, table_attr):
        df = getattr(self, table_attr)
        assert df.count() > 0, f"{table_attr} is empty"

    def test_customers_cdc_has_all_source_rows(self):
        """Bronze should contain all rows from source JSON (before any filtering)."""
        source_count = spark.read.json(f"{VOLUME_PATH}/customers").count()
        bronze_count = self.customers_cdc.count()
        assert bronze_count >= source_count, (
            f"Bronze ({bronze_count}) has fewer rows than source ({source_count})"
        )

    def test_transactions_cdc_has_all_source_rows(self):
        source_count = spark.read.json(f"{VOLUME_PATH}/transactions").count()
        bronze_count = self.transactions_cdc.count()
        assert bronze_count >= source_count, (
            f"Bronze ({bronze_count}) has fewer rows than source ({source_count})"
        )

    def test_customers_cdc_schema(self):
        expected = {"id", "firstname", "lastname", "email", "address", "operation", "operation_date"}
        actual = set(self.customers_cdc.columns)
        missing = expected - actual
        assert not missing, f"Missing bronze columns: {missing}"

In [ ]:
# ============================================================================
# Test: Data Quality Expectations
# ============================================================================

class TestDataQuality(NotebookTestFixture):
    """Validate that DLT expectations filtered bad records between bronze and silver."""

    def run_setup(self):
        self.bronze = spark.table(f"{CATALOG}.{SCHEMA}.customers_cdc")
        self.silver = spark.table(f"{CATALOG}.{SCHEMA}.customers")

    def test_no_null_ids_in_silver(self):
        """expect_or_drop('valid_id', 'id IS NOT NULL') should remove null IDs."""
        null_count = self.silver.filter(col("id").isNull()).count()
        assert null_count == 0, f"Silver contains {null_count} rows with null id"

    def test_source_had_null_ids(self):
        """Confirm the source data actually had null IDs that needed filtering."""
        null_count = self.bronze.filter(col("id").isNull()).count()
        assert null_count > 0, "Source had no null IDs — expectation was never tested"

    def test_silver_row_count_less_than_bronze(self):
        """Silver should have fewer rows than bronze due to expectation drops and dedup."""
        bronze_count = self.bronze.count()
        silver_count = self.silver.count()
        assert silver_count < bronze_count, (
            f"Silver ({silver_count}) should be smaller than bronze ({bronze_count})"
        )

    def test_no_invalid_operations_reached_silver(self):
        """Only APPEND/UPDATE/DELETE should survive expectations.
        Silver is after APPLY CHANGES so operation column is dropped,
        but we can verify via bronze that invalid ops existed."""
        valid_ops = ["APPEND", "DELETE", "UPDATE"]
        invalid_in_bronze = self.bronze.filter(~col("operation").isin(valid_ops)).count()
        assert invalid_in_bronze > 0, "No invalid operations in bronze — expectation untested"

In [ ]:
# ============================================================================
# Test: Silver Layer (Materialized Tables)
# ============================================================================

class TestSilverLayer(NotebookTestFixture):
    """Validate materialized silver tables after APPLY CHANGES."""

    def run_setup(self):
        self.customers = spark.table(f"{CATALOG}.{SCHEMA}.customers")
        self.transactions = spark.table(f"{CATALOG}.{SCHEMA}.transactions")

    def test_customers_not_empty(self):
        assert self.customers.count() > 0, "Silver customers table is empty"

    def test_transactions_not_empty(self):
        assert self.transactions.count() > 0, "Silver transactions table is empty"

    def test_customers_has_no_metadata_columns(self):
        """APPLY CHANGES except_column_list should strip operation/metadata columns."""
        excluded = {"operation", "operation_date", "_rescued_data"}
        present = excluded & set(self.customers.columns)
        assert not present, f"Silver still has metadata columns: {present}"

    def test_transactions_has_no_metadata_columns(self):
        excluded = {"operation", "operation_date", "_rescued_data"}
        present = excluded & set(self.transactions.columns)
        assert not present, f"Silver still has metadata columns: {present}"

    def test_customers_schema(self):
        expected = {"id", "firstname", "lastname", "email", "address"}
        actual = set(self.customers.columns)
        # APPLY CHANGES adds __START_AT etc. for internal tracking; check our columns exist
        missing = expected - actual
        assert not missing, f"Missing silver columns: {missing}"

    def test_customers_ids_are_unique(self):
        """After APPLY CHANGES SCD1, each id should appear exactly once."""
        total = self.customers.count()
        distinct = self.customers.select("id").distinct().count()
        assert total == distinct, (
            f"Duplicate ids in silver: {total} rows but {distinct} distinct ids"
        )

    def test_transactions_ids_are_unique(self):
        total = self.transactions.count()
        distinct = self.transactions.select("id").distinct().count()
        assert total == distinct, (
            f"Duplicate ids in transactions: {total} rows but {distinct} distinct ids"
        )

    def test_deletes_removed_from_silver(self):
        """Records with operation=DELETE in bronze should not appear in silver."""
        bronze = spark.table(f"{CATALOG}.{SCHEMA}.customers_cdc")
        deleted_ids = (
            bronze.filter("operation = 'DELETE' AND id IS NOT NULL")
            .select("id").distinct()
        )
        # Some deleted IDs may have been re-inserted by a later APPEND with a newer timestamp.
        # Find IDs whose latest operation is DELETE.
        from pyspark.sql.window import Window
        from pyspark.sql.functions import row_number
        w = Window.partitionBy("id").orderBy(col("operation_date").desc())
        latest = (
            bronze.filter("id IS NOT NULL AND operation IN ('APPEND','UPDATE','DELETE')")
            .withColumn("_rn", row_number().over(w))
            .filter("_rn = 1")
        )
        truly_deleted = latest.filter("operation = 'DELETE'").select("id")
        still_in_silver = self.customers.join(truly_deleted, "id", "inner").count()
        assert still_in_silver == 0, (
            f"{still_in_silver} deleted records still present in silver"
        )

In [ ]:
# ============================================================================
# Test: SCD Type 2 (Historical Tracking)
# ============================================================================

class TestSCD2(NotebookTestFixture):
    """Validate SCD2_customers table preserves change history."""

    def run_setup(self):
        self.scd2 = spark.table(f"{CATALOG}.{SCHEMA}.SCD2_customers")
        self.silver = spark.table(f"{CATALOG}.{SCHEMA}.customers")

    def test_scd2_not_empty(self):
        assert self.scd2.count() > 0, "SCD2_customers is empty"

    def test_scd2_has_validity_columns(self):
        """SCD2 tables should have __START_AT and __END_AT tracking columns."""
        cols = set(self.scd2.columns)
        assert "__START_AT" in cols, "Missing __START_AT column"
        assert "__END_AT" in cols, "Missing __END_AT column"

    def test_scd2_has_more_rows_than_silver(self):
        """SCD2 stores history, so it should have >= rows compared to SCD1 silver."""
        scd2_count = self.scd2.count()
        silver_count = self.silver.count()
        assert scd2_count >= silver_count, (
            f"SCD2 ({scd2_count}) has fewer rows than silver ({silver_count})"
        )

    def test_scd2_current_records_have_null_end_at(self):
        """The most recent version of each id should have __END_AT = NULL."""
        current = self.scd2.filter(col("__END_AT").isNull())
        assert current.count() > 0, "No current records (all __END_AT are non-null)"

    def test_scd2_current_matches_silver_count(self):
        """Number of current SCD2 rows (end_at IS NULL) should match silver row count."""
        current_scd2 = self.scd2.filter(col("__END_AT").isNull()).count()
        silver_count = self.silver.count()
        assert current_scd2 == silver_count, (
            f"Current SCD2 rows ({current_scd2}) != silver rows ({silver_count})"
        )

    def test_scd2_no_overlapping_validity_periods(self):
        """For any given id, validity periods should not overlap."""
        from pyspark.sql.functions import lead
        from pyspark.sql.window import Window
        w = Window.partitionBy("id").orderBy("__START_AT")
        with_next = self.scd2.withColumn("_next_start", lead("__START_AT").over(w))
        overlaps = with_next.filter(
            col("__END_AT").isNotNull() & col("_next_start").isNotNull() &
            (col("__END_AT") > col("_next_start"))
        ).count()
        assert overlaps == 0, f"{overlaps} SCD2 records have overlapping validity periods"

    def test_scd2_has_no_metadata_columns(self):
        excluded = {"operation", "operation_date", "_rescued_data"}
        present = excluded & set(self.scd2.columns)
        assert not present, f"SCD2 still has metadata columns: {present}"

In [ ]:
# ============================================================================
# Test: Cross-Table Consistency
# ============================================================================

class TestCrossTableConsistency(NotebookTestFixture):
    """Validate referential integrity and consistency across pipeline layers."""

    def run_setup(self):
        self.customers = spark.table(f"{CATALOG}.{SCHEMA}.customers")
        self.transactions = spark.table(f"{CATALOG}.{SCHEMA}.transactions")

    def test_transaction_customer_ids_exist_in_customers(self):
        """Transaction customer_ids should reference valid customer ids."""
        if "customer_id" not in self.transactions.columns:
            return  # customer_id may be stripped by except_column_list
        orphans = (
            self.transactions.select(col("customer_id").alias("id"))
            .join(self.customers.select("id"), "id", "left_anti")
            .count()
        )
        # Some orphans expected due to DELETE operations on customers
        total = self.transactions.count()
        orphan_pct = (orphans / total * 100) if total > 0 else 0
        assert orphan_pct < 20, (
            f"{orphan_pct:.1f}% of transactions reference deleted customers (threshold: 20%)"
        )

    def test_all_silver_ids_are_non_null(self):
        """Both silver tables should have no null primary keys."""
        for name, df in [("customers", self.customers), ("transactions", self.transactions)]:
            nulls = df.filter(col("id").isNull()).count()
            assert nulls == 0, f"{name} has {nulls} null ids"

In [ ]:
# ============================================================================
# Run all tests with NotebookRunner (parallel execution)
# ============================================================================

runner = NotebookRunner(parallel=True)
results = runner.run()
runner.summary()